# Colab EDA Notebook for `/content/archive.zip`

This notebook is Colab-first. It expects the dataset zip at `/content/archive.zip`, extracts it to `/content/archive_eda`, scans the dataset structure, summarizes images and labels, and previews sample images.

## 1. Imports And Paths

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import json
import math
import os
import random
import shutil
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageOps

ZIP_PATH = Path('/content/archive.zip')
EXTRACT_DIR = Path('/content/archive_eda')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
LABEL_EXTS = {'.json', '.csv', '.txt', '.xml'}
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print('ZIP_PATH:', ZIP_PATH)
print('EXTRACT_DIR:', EXTRACT_DIR)

## 2. Extract Archive

In [ ]:
if not ZIP_PATH.exists():
    print(f'Archive not found: {ZIP_PATH}')
    print('Upload archive.zip now, or change ZIP_PATH in the first cell.')
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError(f'No file uploaded. Expected: {ZIP_PATH}')
        uploaded_name = next(iter(uploaded.keys()))
        uploaded_path = Path('/content') / uploaded_name
        if uploaded_path != ZIP_PATH:
            shutil.move(str(uploaded_path), ZIP_PATH)
        print(f'Uploaded archive saved to: {ZIP_PATH}')
    except ImportError:
        raise FileNotFoundError(f'Archive not found: {ZIP_PATH}')

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    members = zf.namelist()
    print(f'Files in zip: {len(members):,}')
    print('First 20 members:')
    for name in members[:20]:
        print(' -', name)

    if not any(EXTRACT_DIR.iterdir()):
        zf.extractall(EXTRACT_DIR)
        print(f'Extracted to: {EXTRACT_DIR}')
    else:
        print(f'Extract dir already has files, skip extraction: {EXTRACT_DIR}')

## 3. Folder Tree And File Types

In [ ]:
def print_tree(root, max_depth=3, max_items_per_dir=12):
    root = Path(root)
    print(root)

    def walk(path, depth):
        if depth > max_depth:
            return
        items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        visible = items[:max_items_per_dir]
        for item in visible:
            prefix = '  ' * depth + ('- ' if item.is_file() else '+ ')
            suffix = f' ({item.stat().st_size / 1024:.1f} KB)' if item.is_file() else ''
            print(prefix + item.name + suffix)
            if item.is_dir():
                walk(item, depth + 1)
        if len(items) > max_items_per_dir:
            print('  ' * depth + f'... {len(items) - max_items_per_dir} more items')

    walk(root, 1)

all_files = [p for p in EXTRACT_DIR.rglob('*') if p.is_file()]
ext_counts = Counter(p.suffix.lower() or '<no_ext>' for p in all_files)

print_tree(EXTRACT_DIR)
print('\nTotal files:', f'{len(all_files):,}')
pd.DataFrame(ext_counts.most_common(), columns=['extension', 'count']).head(30)

## 4. Image Inventory

In [ ]:
image_paths = [p for p in all_files if p.suffix.lower() in IMAGE_EXTS]
label_paths = [p for p in all_files if p.suffix.lower() in LABEL_EXTS]

print('Image files:', f'{len(image_paths):,}')
print('Label-like files:', f'{len(label_paths):,}')

image_ext_df = pd.DataFrame(Counter(p.suffix.lower() for p in image_paths).most_common(), columns=['image_ext', 'count'])
label_ext_df = pd.DataFrame(Counter(p.suffix.lower() for p in label_paths).most_common(), columns=['label_ext', 'count'])

display(image_ext_df)
display(label_ext_df)

## 5. Infer Splits From Paths

In [ ]:
SPLIT_NAMES = {'train', 'training', 'val', 'valid', 'validation', 'test', 'testing'}

def infer_split(path):
    parts = [part.lower() for part in Path(path).parts]
    for part in parts:
        if part in SPLIT_NAMES:
            if part in {'training'}:
                return 'train'
            if part in {'valid', 'validation'}:
                return 'val'
            if part in {'testing'}:
                return 'test'
            return part
    return 'unknown'

split_counts = Counter(infer_split(p) for p in image_paths)
split_df = pd.DataFrame(split_counts.most_common(), columns=['split', 'image_count'])
display(split_df)

## 6. Image Size And Corruption Check

In [ ]:
def inspect_images(paths, max_images=None):
    rows = []
    bad = []
    selected = paths if max_images is None else paths[:max_images]
    for path in selected:
        try:
            with Image.open(path) as img:
                img = ImageOps.exif_transpose(img)
                width, height = img.size
                rows.append({
                    'path': str(path),
                    'file_name': path.name,
                    'split': infer_split(path),
                    'ext': path.suffix.lower(),
                    'width': width,
                    'height': height,
                    'aspect_ratio': width / height if height else None,
                    'megapixels': width * height / 1_000_000,
                    'file_size_kb': path.stat().st_size / 1024,
                    'parent': path.parent.name,
                })
        except Exception as exc:
            bad.append({'path': str(path), 'error': repr(exc)})
    return pd.DataFrame(rows), pd.DataFrame(bad)

image_df, bad_image_df = inspect_images(image_paths)
print('Valid images:', f'{len(image_df):,}')
print('Bad images:', f'{len(bad_image_df):,}')
display(image_df.head())
display(bad_image_df.head())

In [ ]:
if len(image_df):
    summary_cols = ['width', 'height', 'aspect_ratio', 'megapixels', 'file_size_kb']
    display(image_df[summary_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
    display(image_df.groupby('split')[summary_cols].agg(['count', 'mean', 'min', 'max']).round(3))

## 7. Visualize Image Statistics

In [ ]:
if len(image_df):
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    image_df['width'].hist(ax=axes[0], bins=40)
    axes[0].set_title('Width distribution')
    image_df['height'].hist(ax=axes[1], bins=40)
    axes[1].set_title('Height distribution')
    image_df['aspect_ratio'].hist(ax=axes[2], bins=40)
    axes[2].set_title('Aspect ratio distribution')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 6))
    plt.scatter(image_df['width'], image_df['height'], s=8, alpha=0.35)
    plt.xlabel('width')
    plt.ylabel('height')
    plt.title('Image resolution scatter')
    plt.grid(alpha=0.25)
    plt.show()

## 8. Preview Sample Images

In [ ]:
def show_image_grid(paths, n=12, cols=4, title='Sample images'):
    if not paths:
        print('No images to show.')
        return
    sample = random.sample(paths, k=min(n, len(paths)))
    rows = math.ceil(len(sample) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax, path in zip(axes, sample):
        try:
            img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
            ax.imshow(img)
            ax.set_title(f'{infer_split(path)} | {path.name}', fontsize=9)
        except Exception as exc:
            ax.text(0.5, 0.5, repr(exc), ha='center', va='center')
        ax.axis('off')
    for ax in axes[len(sample):]:
        ax.axis('off')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_image_grid(image_paths, n=12, cols=4)

## 9. Label File Overview

In [ ]:
label_rows = []
for path in label_paths:
    label_rows.append({
        'path': str(path),
        'file_name': path.name,
        'ext': path.suffix.lower(),
        'split': infer_split(path),
        'size_kb': path.stat().st_size / 1024,
        'parent': path.parent.name,
    })
label_df = pd.DataFrame(label_rows)
display(label_df.head(20))
if len(label_df):
    display(label_df.groupby(['ext', 'split']).size().reset_index(name='count').sort_values('count', ascending=False))

## 10. JSON Label Scanner

In [ ]:
json_paths = [p for p in label_paths if p.suffix.lower() == '.json']

def summarize_json(path):
    try:
        with open(path, 'r', encoding='utf-8') as handle:
            data = json.load(handle)
        if isinstance(data, dict):
            keys = list(data.keys())[:20]
            return {'path': str(path), 'type': 'dict', 'top_keys': keys, 'length': len(data)}
        if isinstance(data, list):
            sample = data[0] if data else None
            keys = list(sample.keys())[:20] if isinstance(sample, dict) else []
            return {'path': str(path), 'type': 'list', 'top_keys': keys, 'length': len(data)}
        return {'path': str(path), 'type': type(data).__name__, 'top_keys': [], 'length': None}
    except Exception as exc:
        return {'path': str(path), 'type': 'error', 'top_keys': [], 'length': None, 'error': repr(exc)}

json_summary_df = pd.DataFrame([summarize_json(p) for p in json_paths[:50]])
display(json_summary_df)

## 11. CSV Label Scanner

In [ ]:
csv_paths = [p for p in label_paths if p.suffix.lower() == '.csv']
for path in csv_paths[:5]:
    print('\nCSV:', path)
    try:
        df = pd.read_csv(path)
        print('shape:', df.shape)
        display(df.head())
    except Exception as exc:
        print('Could not read CSV:', repr(exc))

## 12. Optional COCO-Style Annotation Summary

In [ ]:
def find_coco_json(paths):
    candidates = []
    for path in paths:
        try:
            with open(path, 'r', encoding='utf-8') as handle:
                data = json.load(handle)
            if isinstance(data, dict) and {'images', 'annotations', 'categories'}.issubset(data.keys()):
                candidates.append(path)
        except Exception:
            pass
    return candidates

coco_paths = find_coco_json(json_paths[:100])
print('COCO-like json files found:', len(coco_paths))
for p in coco_paths[:10]:
    print(' -', p)

if coco_paths:
    coco_path = coco_paths[0]
    with open(coco_path, 'r', encoding='utf-8') as handle:
        coco = json.load(handle)
    categories = {cat['id']: cat.get('name', str(cat['id'])) for cat in coco.get('categories', [])}
    cat_counts = Counter(categories.get(ann.get('category_id'), ann.get('category_id')) for ann in coco.get('annotations', []))
    print('Using:', coco_path)
    print('images:', len(coco.get('images', [])))
    print('annotations:', len(coco.get('annotations', [])))
    display(pd.DataFrame(cat_counts.most_common(), columns=['category', 'annotation_count']).head(30))

## 13. EDA Summary Export

In [ ]:
OUT_DIR = Path('/content/eda_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

if len(image_df):
    image_df.to_csv(OUT_DIR / 'image_inventory.csv', index=False)
if len(label_df):
    label_df.to_csv(OUT_DIR / 'label_inventory.csv', index=False)
if len(bad_image_df):
    bad_image_df.to_csv(OUT_DIR / 'bad_images.csv', index=False)

summary = {
    'zip_path': str(ZIP_PATH),
    'extract_dir': str(EXTRACT_DIR),
    'total_files': len(all_files),
    'image_files': len(image_paths),
    'label_like_files': len(label_paths),
    'bad_images': len(bad_image_df),
    'splits': dict(split_counts),
    'extensions': dict(ext_counts),
}
with open(OUT_DIR / 'eda_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)

print('Saved EDA outputs to:', OUT_DIR)
print(json.dumps(summary, indent=2)[:2000])